In [9]:
# Load cleaned datasets

import pandas as pd
from pathlib import Path

# Path to cleaned datasets
cleaned_folder = Path(
    r"D:\IVY Files\15 Days Data Analytics Sprint\day01-urban-ghost-kitchen\Datasets\Cleaned"
)

# Find cleaned CSV files
csv_files = cleaned_folder.glob("*.csv")

# Store datasets
data = {}

# Load each CSV file
for file in csv_files:
    dataset_name = file.stem
    data[dataset_name] = pd.read_csv(file)

# Check whether datasets were loaded
if data:
    print("Cleaned datasets loaded successfully")
else:
    print("Unable to load cleaned datasets. Try again later!")

Cleaned datasets loaded successfully


In [10]:
# Check loaded datasets

for name, df in data.items():
    print(f"{name}: {df.shape}")

customers_clean: (1000, 4)
kitchens_clean: (25, 5)
menu_items_clean: (45000, 6)
orders_clean: (20000, 10)


In [11]:
# Inspect cleaned datasets

for name, df in data.items():
    print("\n" + "=" * 50)
    print(name.upper())
    print("=" * 50)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())


CUSTOMERS_CLEAN
Shape: (1000, 4)

Columns:
['customer_id', 'city', 'signup_date', 'customer_segment']

KITCHENS_CLEAN
Shape: (25, 5)

Columns:
['kitchen_id', 'kitchen_name', 'zone', 'monthly_rent', 'staff_count']

MENU_ITEMS_CLEAN
Shape: (45000, 6)

Columns:
['item_id', 'order_id', 'item_name', 'category', 'quantity', 'unit_price']

ORDERS_CLEAN
Shape: (20000, 10)

Columns:
['order_id', 'customer_id', 'kitchen_id', 'order_date', 'order_value', 'discount', 'delivery_fee', 'delivery_distance_km', 'delivery_time_min', 'order_status']


In [12]:
# Check data types

for name, df in data.items():
    print("\n" + "=" * 50)
    print(name.upper())
    print("=" * 50)

    print(df.dtypes)


CUSTOMERS_CLEAN
customer_id         str
city                str
signup_date         str
customer_segment    str
dtype: object

KITCHENS_CLEAN
kitchen_id        str
kitchen_name      str
zone              str
monthly_rent    int64
staff_count     int64
dtype: object

MENU_ITEMS_CLEAN
item_id           str
order_id          str
item_name         str
category          str
quantity        int64
unit_price    float64
dtype: object

ORDERS_CLEAN
order_id                    str
customer_id                 str
kitchen_id                  str
order_date                  str
order_value             float64
discount                float64
delivery_fee            float64
delivery_distance_km    float64
delivery_time_min       float64
order_status                str
dtype: object


In [15]:
# Observed that signup_date(customer_clean table) in str and order_date(orders_clean) table.
# Turn those datatypes to dateformat

# Convert date columns from string to datetime format

date_columns = {
    "customers_clean": "signup_date",
    "orders_clean": "order_date"
}

for dataset, column in date_columns.items():
    data[dataset][column] = pd.to_datetime(
        data[dataset][column],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

print("Date columns converted successfully")

Date columns converted successfully


In [16]:
# Check date column datatypes

print("signup_date:", data["customers_clean"]["signup_date"].dtype)
print("order_date:", data["orders_clean"]["order_date"].dtype)

signup_date: datetime64[us]
order_date: datetime64[us]


In [17]:
# Create time-based features

orders = data["orders_clean"]

orders["order_year"] = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month
orders["order_month_name"] = orders["order_date"].dt.month_name()
orders["order_day"] = orders["order_date"].dt.day
orders["order_day_name"] = orders["order_date"].dt.day_name()
orders["order_hour"] = orders["order_date"].dt.hour
orders["is_weekend"] = orders["order_date"].dt.dayofweek >= 5

print("Time-based features created successfully")

Time-based features created successfully


In [18]:
# Check time-based features

print(
    orders[
        [
            "order_date",
            "order_year",
            "order_month",
            "order_month_name",
            "order_day",
            "order_day_name",
            "order_hour",
            "is_weekend"
        ]
    ].head()
)

           order_date  order_year  order_month order_month_name  order_day  \
0 2025-03-27 13:56:00        2025            3            March         27   
1 2025-06-14 21:35:00        2025            6             June         14   
2 2025-10-03 15:06:00        2025           10          October          3   
3 2025-11-21 09:46:00        2025           11         November         21   
4 2025-06-07 18:56:00        2025            6             June          7   

  order_day_name  order_hour  is_weekend  
0       Thursday          13       False  
1       Saturday          21        True  
2         Friday          15       False  
3         Friday           9       False  
4       Saturday          18        True  


In [21]:
# Create Order Level Features


orders = data["orders_clean"]

orders["gross_amount"] = (
    orders["order_value"] + orders["discount"]
)

orders["total_customer_charge"] = (
    orders["order_value"]
    + orders["delivery_fee"]
    - orders["discount"]
)

orders["discount_percentage"] = (
    orders["discount"]
    / orders["order_value"]
) * 100

orders["delivery_cost_per_km"] = (
    orders["delivery_fee"]
    / orders["delivery_distance_km"]
)

orders["delivery_speed_km_per_min"] = (
    orders["delivery_distance_km"]
    / orders["delivery_time_min"]
)

print("Order-level features created successfully")

Order-level features created successfully


In [22]:
# Check order-level features

print(
    orders[
        [
            "order_id",
            "order_value",
            "discount",
            "delivery_fee",
            "delivery_distance_km",
            "delivery_time_min",
            "gross_amount",
            "total_customer_charge",
            "discount_percentage",
            "delivery_cost_per_km",
            "delivery_speed_km_per_min"
        ]
    ].head()
)

     order_id  order_value  discount  delivery_fee  delivery_distance_km  \
0  ORD-000001       419.48     13.93         34.73                  2.55   
1  ORD-000002       710.98     12.65         38.14                  3.39   
2  ORD-000003       356.76      4.16         23.35                  0.50   
3  ORD-000004       644.31     12.40         27.30                  1.09   
4  ORD-000005       195.31      0.00         22.99                  0.50   

   delivery_time_min  gross_amount  total_customer_charge  \
0               19.4        433.41                 440.28   
1               24.1        723.63                 736.47   
2               10.0        360.92                 375.95   
3               18.1        656.71                 659.21   
4               13.7        195.31                 218.30   

   discount_percentage  delivery_cost_per_km  delivery_speed_km_per_min  
0             3.320778             13.619608                   0.131443  
1             1.779234      

In [23]:
# Create customer-level features

orders = data["orders_clean"]

customer_features = (
    orders.groupby("customer_id")
    .agg(
        total_orders=("order_id", "count"),
        total_spend=("order_value", "sum"),
        average_order_value=("order_value", "mean"),
        total_discount=("discount", "sum"),
        total_delivery_fee=("delivery_fee", "sum"),
        average_delivery_time=("delivery_time_min", "mean")
    )
    .reset_index()
)

# Count orders by status
status_counts = (
    orders.pivot_table(
        index="customer_id",
        columns="order_status",
        values="order_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

# Rename status columns
status_counts = status_counts.rename(
    columns={
        "Delivered": "delivered_orders",
        "Cancelled": "cancelled_orders",
        "Refunded": "refunded_orders"
    }
)

# Merge customer features
customer_features = customer_features.merge(
    status_counts,
    on="customer_id",
    how="left"
)

print("Customer-level features created successfully")

Customer-level features created successfully


In [25]:
# Check customer-level features

display(customer_features.head())
display("\nShape:", customer_features.shape)

,customer_id,total_orders,total_spend,average_order_value,total_discount,total_delivery_fee,average_delivery_time,cancelled_orders,delivered_orders,refunded_orders
0,CUS-00001,10,6000.34,600.034000,61.03,491.91,26.750000,0,10,0
1,CUS-00002,54,31030.97,574.647593,1075.74,2644.67,30.244444,6,46,2
2,CUS-00003,73,45263.81,620.052192,2144.83,3589.41,28.752055,6,65,2
3,CUS-00004,21,11247.92,535.615238,354.68,1043.92,29.971429,4,17,0
4,CUS-00005,12,8937.86,744.821667,313.65,413.61,21.691667,1,11,0


'\nShape:'

(980, 10)

In [26]:
# Create kitchen-level features

orders = data["orders_clean"]
kitchens = data["kitchens_clean"]

kitchen_features = (
    orders.groupby("kitchen_id")
    .agg(
        total_orders=("order_id", "count"),
        total_revenue=("order_value", "sum"),
        average_order_value=("order_value", "mean"),
        average_delivery_time=("delivery_time_min", "mean"),
        average_delivery_distance=("delivery_distance_km", "mean")
    )
    .reset_index()
)

# Count orders by status
kitchen_status = (
    orders.pivot_table(
        index="kitchen_id",
        columns="order_status",
        values="order_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

# Rename status columns
kitchen_status = kitchen_status.rename(
    columns={
        "Delivered": "delivered_orders",
        "Cancelled": "cancelled_orders",
        "Refunded": "refunded_orders"
    }
)

# Merge status metrics
kitchen_features = kitchen_features.merge(
    kitchen_status,
    on="kitchen_id",
    how="left"
)

# Add kitchen information
kitchen_features = kitchen_features.merge(
    kitchens,
    on="kitchen_id",
    how="left"
)

print("Kitchen-level features created successfully")

Kitchen-level features created successfully


In [28]:
# Check kitchen-level features

display(kitchen_features.head())
display("\nShape:", kitchen_features.shape)

,kitchen_id,total_orders,total_revenue,average_order_value,average_delivery_time,average_delivery_distance,cancelled_orders,delivered_orders,refunded_orders,kitchen_name,zone,monthly_rent,staff_count
0,KIT-001,1353,776258.51,573.731345,26.674427,4.836888,247,1055,51,Tasty Trail Central,Central Zone,73989,16
1,KIT-002,3427,1949309.57,568.809329,26.767639,4.911727,284,3016,127,Dish Dock Central,Central Zone,76733,19
2,KIT-003,778,445771.87,572.971555,23.440488,3.930566,79,666,33,Cravings Corner Tech,Tech Park Belt,113123,11
3,KIT-004,344,197357.10,573.712500,34.583721,7.359855,31,299,14,Grill Grove Old,Old City,210936,7
4,KIT-005,774,446962.69,577.471176,23.213307,3.806873,131,609,34,Bite Bazaar South,South Zone,91696,12


'\nShape:'

(25, 13)

In [30]:
# Create menu-item features

menu_items = data["menu_items_clean"]

menu_features = (
    menu_items.groupby(["item_id", "item_name", "category"])
    .agg(
        total_quantity_sold=("quantity", "sum"),
        total_sales=("unit_price", lambda x: (x * menu_items.loc[x.index, "quantity"]).sum()),
        average_unit_price=("unit_price", "mean"),
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)

print("Menu-item features created successfully")

Menu-item features created successfully


In [33]:
# Check menu-item features

display(menu_features.head())
display("Shape:", menu_features.shape)

,item_id,item_name,category,total_quantity_sold,total_sales,average_unit_price,total_orders
0,ITM-000001,Schezwan Noodles,Noodles,4,968.28,242.07,1
1,ITM-000002,Mutton Biryani,Biryani,2,570.10,285.05,1
2,ITM-000003,Chicken Zinger Burger,Burger,1,232.88,232.88,1
3,ITM-000004,Singapore Noodles,Noodles,1,275.71,275.71,1
4,ITM-000005,Egg Biryani,Biryani,3,885.84,295.28,1


'Shape:'

(45000, 7)

In [34]:
# Create delivery-performance features

orders = data["orders_clean"]

if not orders.empty:

    orders["delivery_time_category"] = pd.cut(
        orders["delivery_time_min"],
        bins=[0, 30, 60, float("inf")],
        labels=["Fast", "Moderate", "Slow"]
    )

    orders["distance_category"] = pd.cut(
        orders["delivery_distance_km"],
        bins=[0, 5, 10, float("inf")],
        labels=["Short", "Medium", "Long"]
    )

    orders["delivery_cost_per_km"] = (
        orders["delivery_fee"]
        / orders["delivery_distance_km"]
    )

    orders["delivery_efficiency"] = (
        orders["delivery_distance_km"]
        / orders["delivery_time_min"]
    )

    print("Delivery-performance features created successfully")

else:
    print("Unable to create delivery features. Dataset is empty!")

Delivery-performance features created successfully


In [36]:
# Check delivery-performance features

display(
    orders[
        [
            "order_id",
            "delivery_time_min",
            "delivery_distance_km",
            "delivery_fee",
            "delivery_time_category",
            "distance_category",
            "delivery_cost_per_km",
            "delivery_efficiency"
        ]
    ].head()
)

,order_id,delivery_time_min,delivery_distance_km,delivery_fee,delivery_time_category,distance_category,delivery_cost_per_km,delivery_efficiency
0,ORD-000001,19.4,2.55,34.73,Fast,Short,13.619608,0.131443
1,ORD-000002,24.1,3.39,38.14,Fast,Short,11.250737,0.140664
2,ORD-000003,10.0,0.50,23.35,Fast,Short,46.700000,0.050000
3,ORD-000004,18.1,1.09,27.30,Fast,Short,25.045872,0.060221
4,ORD-000005,13.7,0.50,22.99,Fast,Short,45.980000,0.036496


In [37]:
# Create business KPIs

orders = data["orders_clean"]

if not orders.empty:

    total_orders = orders["order_id"].nunique()

    total_revenue = orders["order_value"].sum()

    average_order_value = orders["order_value"].mean()

    total_discount = orders["discount"].sum()

    total_delivery_fee = orders["delivery_fee"].sum()

    average_delivery_time = orders["delivery_time_min"].mean()

    average_delivery_distance = orders["delivery_distance_km"].mean()

    cancelled_orders = (
        orders["order_status"] == "Cancelled"
    ).sum()

    refunded_orders = (
        orders["order_status"] == "Refunded"
    ).sum()

    delivered_orders = (
        orders["order_status"] == "Delivered"
    ).sum()

    cancellation_rate = (
        cancelled_orders / total_orders
    ) * 100

    refund_rate = (
        refunded_orders / total_orders
    ) * 100

    delivery_rate = (
        delivered_orders / total_orders
    ) * 100

    print("Business KPIs created successfully")

else:
    print("Unable to create business KPIs. Dataset is empty!")

Business KPIs created successfully


In [38]:
# Display business KPIs

kpis = {
    "Total Orders": total_orders,
    "Total Revenue": total_revenue,
    "Average Order Value": average_order_value,
    "Total Discount": total_discount,
    "Total Delivery Fee": total_delivery_fee,
    "Average Delivery Time": average_delivery_time,
    "Average Delivery Distance": average_delivery_distance,
    "Cancellation Rate (%)": cancellation_rate,
    "Refund Rate (%)": refund_rate,
    "Delivery Rate (%)": delivery_rate
}

for name, value in kpis.items():
    print(f"{name}: {value:.2f}")

Total Orders: 20000.00
Total Revenue: 11507673.68
Average Order Value: 575.38
Total Discount: 370176.43
Total Delivery Fee: 1027796.64
Average Delivery Time: 29.31
Average Delivery Distance: 5.21
Cancellation Rate (%): 10.19
Refund Rate (%): 4.00
Delivery Rate (%): 85.81


In [39]:
# Check engineered columns

print("Orders columns:")
print(data["orders_clean"].columns.tolist())

print("\nCustomer features:")
print(customer_features.columns.tolist())

print("\nKitchen features:")
print(kitchen_features.columns.tolist())

print("\nMenu features:")
print(menu_features.columns.tolist())

# Check missing values in engineered data

print("Orders:")
print(data["orders_clean"].isna().sum())

print("\nCustomer features:")
print(customer_features.isna().sum())

print("\nKitchen features:")
print(kitchen_features.isna().sum())

print("\nMenu features:")
print(menu_features.isna().sum())

# Check numeric feature values

print("Order-level feature summary:")
print(
    data["orders_clean"][
        [
            "gross_amount",
            "total_customer_charge",
            "discount_percentage",
            "delivery_cost_per_km",
            "delivery_speed_km_per_min"
        ]
    ].describe()
)

# Check delivery buckets

print(
    data["orders_clean"]["delivery_time_category"]
    .value_counts(dropna=False)
)

print(
    data["orders_clean"]["distance_category"]
    .value_counts(dropna=False)
)

Orders columns:
['order_id', 'customer_id', 'kitchen_id', 'order_date', 'order_value', 'discount', 'delivery_fee', 'delivery_distance_km', 'delivery_time_min', 'order_status', 'order_year', 'order_month', 'order_month_name', 'order_day', 'order_day_name', 'order_hour', 'is_weekend', 'gross_amount', 'total_customer_charge', 'discount_percentage', 'delivery_cost_per_km', 'delivery_speed_km_per_min', 'delivery_time_category', 'distance_category', 'delivery_efficiency']

Customer features:
['customer_id', 'total_orders', 'total_spend', 'average_order_value', 'total_discount', 'total_delivery_fee', 'average_delivery_time', 'cancelled_orders', 'delivered_orders', 'refunded_orders']

Kitchen features:
['kitchen_id', 'total_orders', 'total_revenue', 'average_order_value', 'average_delivery_time', 'average_delivery_distance', 'cancelled_orders', 'delivered_orders', 'refunded_orders', 'kitchen_name', 'zone', 'monthly_rent', 'staff_count']

Menu features:
['item_id', 'item_name', 'category', 'tot

In [40]:
# Export analysis-ready datasets

from pathlib import Path

analysis_folder = Path(
    r"D:\IVY Files\15 Days Data Analytics Sprint\day01-urban-ghost-kitchen\Datasets\Analysis_Ready"
)

analysis_folder.mkdir(parents=True, exist_ok=True)

print("Analysis-ready folder ready")

# Save analysis-ready datasets

data["orders_clean"].to_csv(
    analysis_folder / "orders_analysis.csv",
    index=False
)

customer_features.to_csv(
    analysis_folder / "customer_features.csv",
    index=False
)

kitchen_features.to_csv(
    analysis_folder / "kitchen_features.csv",
    index=False
)

menu_features.to_csv(
    analysis_folder / "menu_features.csv",
    index=False
)

print("Analysis-ready datasets exported successfully")

Analysis-ready folder ready
Analysis-ready datasets exported successfully
